# 08 · Sparse autoencoders: pulling interpretable features out of superposition

**Track:** Training LLMs & Frontier Research · **Runtime (CPU default):** ~2.5 minutes · **GPU:** optional (bigger dictionary)

Individual neurons in language models are usually **polysemantic**: one MLP neuron might fire on Python keywords,
on French, and on the word "Thursday". The **superposition hypothesis** (Elhage et al., 2022) explains why: a model
can represent many more sparse features than it has neurons by storing them as *almost-orthogonal directions*.
If that is right, the fix is to learn an overcomplete, sparse basis for the activations, a **dictionary**, and look at
its directions instead of the neurons. That is what a **sparse autoencoder (SAE)** does. Anthropic's *Towards
Monosemanticity* (2023) trained SAEs on a 1-layer transformer's MLP and found thousands of crisp features; *Scaling
Monosemanticity* (2024) did it on Claude 3 Sonnet; OpenAI and Google DeepMind followed with TopK, Gated and JumpReLU
SAEs and open SAE suites. Here we train two small SAEs (the classic ReLU+L1 and the TopK variant) on the MLP
activations of a pretrained 1-layer model, evaluate them the way the papers do, and inspect what the features
respond to.

> **Source & license credit.** Built on **[TransformerLens](https://github.com/TransformerLensOrg/TransformerLens)**
> (MIT). Model: Neel Nanda's 1-layer GELU toy model
> **[NeelNanda/GELU_1L512W_C4_Code](https://huggingface.co/NeelNanda/GELU_1L512W_C4_Code)** (trained on C4 + code;
> downloaded at runtime, not redistributed; see the model card). SAE architectures and training choices follow the
> cited papers (Bricken et al. 2023; Gao et al. 2024). The code is an original, minimal implementation. For
> production SAEs see [SAELens](https://github.com/jbloomAus/SAELens) (MIT) and
> [Gemma Scope](https://huggingface.co/google/gemma-scope). Activation data: the public-domain Shakespeare excerpt plus
> this repo's own Python source in `research/src/`. See [`research/THIRD_PARTY_NOTICES.md`](../THIRD_PARTY_NOTICES.md).

## Learning goals

1. Explain superposition and why a **sparse, overcomplete** dictionary can undo it.
2. Implement a ReLU SAE with an L1 sparsity penalty and unit-norm decoder (Bricken et al. 2023), and a **TopK SAE**
   (Gao et al. 2024).
3. Evaluate SAEs on the standard trade-offs: **L0** (active latents per token), **fraction of variance explained**,
   **dead latents**, and **downstream CE loss recovered** when the SAE reconstruction is spliced into the model.
4. Interpret latents with **max-activating examples**, and check for features specific to code vs. prose.
5. Be explicit about what a 3-minute CPU SAE can and cannot show.

## Background

* Elhage et al., *Toy Models of Superposition*, Anthropic 2022. [arXiv:2209.10652](https://arxiv.org/abs/2209.10652)
* Bricken, Templeton, Batson, et al., *Towards Monosemanticity: Decomposing Language Models With Dictionary Learning*,
  Anthropic 2023. [transformer-circuits.pub/2023/monosemantic-features](https://transformer-circuits.pub/2023/monosemantic-features/index.html)
* Templeton, Conerly, Marcus, et al., *Scaling Monosemanticity: Extracting Interpretable Features from Claude 3 Sonnet*,
  Anthropic 2024. [transformer-circuits.pub/2024/scaling-monosemanticity](https://transformer-circuits.pub/2024/scaling-monosemanticity/index.html)
* Cunningham, Ewart, Riggs, Huben, Sharkey, *Sparse Autoencoders Find Highly Interpretable Features in Language Models*,
  2023. [arXiv:2309.08600](https://arxiv.org/abs/2309.08600)
* Gao, la Tour, Tillman, Goh, Troll, Radford, Sutskever, Leike, Wu, *Scaling and evaluating sparse autoencoders*,
  OpenAI 2024. [arXiv:2406.04093](https://arxiv.org/abs/2406.04093) (TopK SAEs, scaling laws for SAEs)
* Rajamanoharan et al., *Improving Dictionary Learning with Gated Sparse Autoencoders*, Google DeepMind 2024.
  [arXiv:2404.16014](https://arxiv.org/abs/2404.16014)
* Rajamanoharan et al., *Jumping Ahead: Improving Reconstruction Fidelity with JumpReLU Sparse Autoencoders*,
  Google DeepMind 2024. [arXiv:2407.14435](https://arxiv.org/abs/2407.14435)
* Lieberum et al., *Gemma Scope: Open Sparse Autoencoders Everywhere All At Once on Gemma 2*, Google DeepMind 2024.
  [arXiv:2408.05147](https://arxiv.org/abs/2408.05147)

### The model

For an activation $x \in \mathbb{R}^{d}$ (here $d = 2048$ MLP neurons) and a dictionary of $m > d$ latents:
$$f(x) = \mathrm{ReLU}\big(W_{enc}(x - b_{dec}) + b_{enc}\big) \in \mathbb{R}^{m}, \qquad \hat{x} = W_{dec} f(x) + b_{dec}.$$
* **ReLU + L1** (Bricken et al.): $\mathcal{L} = \lVert x - \hat{x}\rVert_2^2 + \lambda \lVert f(x)\rVert_1$, with decoder
  columns constrained to unit norm, so the L1 penalty can't be dodged by shrinking $f$ and growing $W_{dec}$.
* **TopK** (Gao et al.): keep only the $k$ largest pre-activations, $f(x) = \mathrm{ReLU}(\mathrm{TopK}(\cdot))$, and
  train on reconstruction alone. Sparsity is set directly ($L_0 = k$), with no L1 "shrinkage" of the active values.

In [ ]:
# --- Setup: make research/src importable (from research/, research/notebooks/, or a Colab clone) ---
import sys, pathlib
for _p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]:
    _src = next((c for c in (_p / "src", _p / "research" / "src") if (c / "lab_config.py").exists()), None)
    if _src:
        sys.path.insert(0, str(_src))
        break
else:
    raise RuntimeError("Could not find research/src -- run this notebook from inside the repo.")

from lab_config import CFG, pick, set_seed, heavy_cell, read_text, quiet_hf_logging

FULL_SIZE = CFG.full_size   # <- set True here (or export LAB_FULL_SIZE=1) for the full-size run
CFG.full_size = FULL_SIZE
set_seed()
print(CFG.summary())
quiet_hf_logging()

In [ ]:
import time, glob
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from transformer_lens.model_bridge import TransformerBridge

model = TransformerBridge.boot_tl_legacy("NeelNanda/GELU_1L512W_C4_Code", device=CFG.device)
model.eval()
HOOK = "blocks.0.mlp.hook_post"                      # the MLP neuron activations, as in Towards Monosemanticity
D_MLP = model.cfg.d_mlp
print(f"model: {model.cfg.n_layers} layer, d_model={model.cfg.d_model}, d_mlp={D_MLP}, act_fn={model.cfg.act_fn}")

## 1 · Collect activations

We tokenize two kinds of text, prose (Shakespeare) and code (this repo's own `research/src/*.py`), into
128-token sequences, run the model up to the MLP (`stop_at_layer=1`, so we skip the expensive 48k-way unembedding), and
store the 2048-d MLP activation at every token. We keep a source label per token so we can later ask whether some
features are code-specific. The last 10% of sequences are held out for evaluation.

In [ ]:
SEQ = 128
N_TOKENS = pick(64_000, 200_000)
tok = model.tokenizer
bos = tok.bos_token_id

prose_ids = tok(read_text("tiny_shakespeare.txt"))["input_ids"]
code_text = "\n\n".join(pathlib.Path(p).read_text() for p in sorted(glob.glob(str(CFG.data_dir.parent / "src" / "*.py"))))
code_ids = tok(code_text)["input_ids"]

def chunk(ids, label):
    n = len(ids) // (SEQ - 1)
    seqs = torch.tensor(ids[: n * (SEQ - 1)]).view(n, SEQ - 1)
    return torch.cat([torch.full((n, 1), bos), seqs], 1), torch.full((n,), label)

code_seqs, code_lab = chunk(code_ids, 1)
prose_seqs, prose_lab = chunk(prose_ids, 0)
n_prose = max(0, N_TOKENS // SEQ - len(code_seqs))
seqs = torch.cat([code_seqs, prose_seqs[:n_prose]])
labels = torch.cat([code_lab, prose_lab[:n_prose]])
perm = torch.randperm(len(seqs), generator=torch.Generator().manual_seed(CFG.seed))
seqs, labels = seqs[perm], labels[perm]
n_eval = max(8, len(seqs) // 10)
train_seqs, eval_seqs = seqs[n_eval:], seqs[:n_eval]
train_lab = labels[n_eval:]
print(f"{len(seqs)} sequences of {SEQ} tokens ({int(labels.sum())} code, {int((1 - labels).sum())} prose); {n_eval} held out")

@torch.no_grad()
def collect(batch_seqs, bs=64):
    out = []
    for i in range(0, len(batch_seqs), bs):
        _, cache = model.run_with_cache(batch_seqs[i:i + bs].to(CFG.device), names_filter=lambda n: n == HOOK, stop_at_layer=1)
        out.append(cache[HOOK][:, 1:].reshape(-1, D_MLP).float().cpu())   # drop the BOS position (an outlier)
    return torch.cat(out)

t0 = time.time()
acts = collect(train_seqs)
tok_ids = train_seqs[:, 1:].reshape(-1)
tok_is_code = train_lab[:, None].expand(-1, SEQ - 1).reshape(-1).bool()
print(f"activations: {tuple(acts.shape)} ({acts.numel() * 4 / 1e6:.0f} MB) in {time.time() - t0:.1f}s")

# Normalise so that E[||x||^2] = d (a common convention; makes the L1 coefficient transferable across layers/models)
SCALE = float((D_MLP / acts.pow(2).sum(-1).mean()).sqrt())
acts *= SCALE
print(f"scale factor {SCALE:.3f}; neurons active (>0) per token: {(acts > 0).float().sum(-1).mean():.0f} of {D_MLP}")

GELU neurons are "dense": hundreds of the 2,048 are non-zero on every token, and each one mixes many unrelated
inputs. An SAE replaces this dense code with a sparse one, ideally with each latent firing for one recognisable reason.

## 2 · The sparse autoencoder

In [ ]:
class SparseAutoencoder(nn.Module):
    def __init__(self, d_in, d_sae, k=None):
        super().__init__()
        self.k = k
        W_dec = torch.randn(d_sae, d_in)
        W_dec /= W_dec.norm(dim=1, keepdim=True)              # unit-norm dictionary directions
        self.W_dec = nn.Parameter(W_dec)
        self.W_enc = nn.Parameter(W_dec.T.clone())            # tied initialisation (Gao et al. 2024)
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.b_dec = nn.Parameter(torch.zeros(d_in))

    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        if self.k is None:
            return torch.relu(pre)
        vals, idx = pre.topk(self.k, dim=-1)
        return torch.zeros_like(pre).scatter(-1, idx, torch.relu(vals))

    def decode(self, f):
        return f @ self.W_dec + self.b_dec

    def forward(self, x):
        f = self.encode(x)
        return self.decode(f), f

    @torch.no_grad()
    def normalize_decoder(self):
        self.W_dec.data /= self.W_dec.data.norm(dim=1, keepdim=True)


def train_sae(sae, data, steps, batch=1024, lr=1e-3, l1=0.0, l1_warmup=0.1, seed=0, log_every=100):
    sae.b_dec.data = data[:20_000].mean(0).clone()           # start the bias at the data mean
    opt = torch.optim.Adam(sae.parameters(), lr=lr, betas=(0.9, 0.999))
    g = torch.Generator().manual_seed(seed)
    log = []
    for step in range(steps):
        x = data[torch.randint(len(data), (batch,), generator=g)]
        x_hat, f = sae(x)
        mse = (x_hat - x).pow(2).sum(-1).mean() / x.shape[-1]   # per-dimension MSE
        lam = l1 * min(1.0, step / max(1, l1_warmup * steps))   # warm up the penalty (fewer dead latents)
        loss = mse + lam * f.sum(-1).mean()                     # f >= 0, so sum(f) = ||f||_1
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        sae.normalize_decoder()
        if step % log_every == 0 or step == steps - 1:
            log.append((step, mse.item(), (f > 0).float().sum(-1).mean().item()))
    return log

D_SAE = pick(4096, 16384)          # 2x / 8x expansion of the 2048 MLP neurons
STEPS = pick(600, 5000)
K = 32
L1 = 2e-3
print(f"dictionary size {D_SAE} ({D_SAE // D_MLP}x), {STEPS} steps x 1024 tokens")

In [ ]:
saes, logs = {}, {}
for name, k, l1 in [("ReLU+L1", None, L1), (f"TopK (k={K})", K, 0.0)]:
    set_seed(CFG.seed)
    sae = SparseAutoencoder(D_MLP, D_SAE, k=k).to("cpu")
    t0 = time.time()
    logs[name] = train_sae(sae, acts, STEPS, l1=l1, seed=CFG.seed)
    saes[name] = sae.eval()
    print(f"{name:12} trained in {time.time() - t0:.0f}s; final batch MSE {logs[name][-1][1]:.4f}, L0 {logs[name][-1][2]:.1f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
for name, lg in logs.items():
    s, mse, l0 = zip(*lg)
    ax[0].plot(s, mse, "o-", ms=3, label=name); ax[1].plot(s, l0, "o-", ms=3, label=name)
ax[0].set(xlabel="step", ylabel="MSE (per dim)", yscale="log", title="reconstruction")
ax[1].set(xlabel="step", ylabel="L0", yscale="log", title="active latents per token")
for a in ax:
    a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3 · Evaluate like the papers do

* **L0**: mean number of active latents per token (lower = sparser, easier to interpret).
* **Fraction of variance explained (FVE)**: $1 - \frac{\sum\lVert x-\hat x\rVert^2}{\sum\lVert x-\bar x\rVert^2}$.
* **Dead latents**: never active on the held-out tokens. Wasted capacity, and a known ReLU-SAE failure mode.
* **CE loss recovered**: replace the MLP activations with the SAE reconstruction *inside the model* and measure the
  language-modelling loss. We report $\frac{L_{\text{zero-abl}} - L_{\text{SAE}}}{L_{\text{zero-abl}} - L_{\text{clean}}}$,
  where zero-ablation sets the MLP activations to 0. Good reconstructions in MSE can still be bad for the model, so
  this is the metric that matters most.

In [ ]:
eval_acts = collect(eval_seqs) * SCALE

@torch.no_grad()
def lm_loss(fwd_hooks=()):
    logits = model.run_with_hooks(eval_seqs.to(CFG.device), fwd_hooks=list(fwd_hooks))
    lp = logits.log_softmax(-1)[:, :-1].gather(-1, eval_seqs[:, 1:, None].to(CFG.device)).squeeze(-1)
    return -lp.mean().item()

def splice(sae):
    def hook_fn(act, hook):
        flat = act.reshape(-1, D_MLP).float() * SCALE
        recon = (sae(flat.cpu())[0].to(act.device) / SCALE).reshape(act.shape).to(act.dtype)
        recon[:, 0] = act[:, 0]          # the SAE never saw BOS (an outlier); keep it untouched
        return recon
    return [(HOOK, hook_fn)]

L_clean = lm_loss()
L_zero = lm_loss([(HOOK, lambda act, hook: torch.zeros_like(act))])
print(f"LM loss: clean {L_clean:.3f} | MLP zero-ablated {L_zero:.3f}")

rows = []
with torch.no_grad():
    for name, sae in saes.items():
        x_hat, f = sae(eval_acts)
        fve = 1 - (x_hat - eval_acts).pow(2).sum() / (eval_acts - eval_acts.mean(0)).pow(2).sum()
        L_sae = lm_loss(splice(sae))
        rows.append(dict(sae=name, L0=(f > 0).float().sum(-1).mean().item(), FVE=fve.item(),
                         dead_frac=((f > 0).sum(0) == 0).float().mean().item(), LM_loss=L_sae,
                         CE_recovered=(L_zero - L_sae) / (L_zero - L_clean)))
import pandas as pd
results = pd.DataFrame(rows).set_index("sae").round(3)
results

**Reading the table.** Compare the two SAEs at their (different) sparsity levels rather than declaring a winner:
the L1 coefficient and $k$ set different points on the sparsity/fidelity frontier. Gao et al. (2024) argue that TopK
gives a better frontier and far fewer dead latents than ReLU+L1 at scale. To test that claim properly you would sweep
λ and k and plot FVE (or CE recovered) against L0 (see *Your turn*). Don't over-read a high CE-recovered number
here: the held-out set is tiny (≈6k tokens) and drawn from the same two sources as the training data, the model has
only one MLP, and zero-ablation is a weak baseline. Our SAEs saw ~0.6M token-samples, while published SAEs see
billions, on far more diverse text.

## 4 · What do the latents respond to?

### 4a · Feature density

Bricken et al. plot a histogram of log₁₀(firing frequency) per latent. A healthy dictionary has most latents firing
rarely but not never. A spike at the far left is dead latents; very frequent latents are often uninterpretable.

In [ ]:
@torch.no_grad()
def latent_stats(sae, data, is_code, n_top=5, chunk=8192):
    """Per-latent firing counts (overall / code / prose) and top activations, computed in chunks to save RAM."""
    m = sae.W_dec.shape[0]
    fire, fire_code = torch.zeros(m), torch.zeros(m)
    top_v, top_i = torch.full((n_top, m), -1.0), torch.zeros((n_top, m), dtype=torch.long)
    for start in range(0, len(data), chunk):
        F = sae.encode(data[start:start + chunk])
        on = F > 0
        fire += on.sum(0); fire_code += on[is_code[start:start + chunk]].sum(0)
        v, i = torch.cat([top_v, F]), torch.cat([top_i, torch.arange(start, start + len(F))[:, None].expand(-1, m)])
        best = v.topk(n_top, dim=0)
        top_v, top_i = best.values, i.gather(0, best.indices)
    n_code = int(is_code.sum()); n_prose = len(data) - n_code
    return dict(freq=fire / len(data), code_rate=fire_code / max(1, n_code),
                prose_rate=(fire - fire_code) / max(1, n_prose), count=fire, top_v=top_v, top_i=top_i)

stats = {name: latent_stats(sae, acts, tok_is_code) for name, sae in saes.items()}

fig, ax = plt.subplots(figsize=(6, 3))
for name, st in stats.items():
    ax.hist(np.log10(st["freq"].numpy() + 1e-7), bins=60, alpha=0.6, label=name)
ax.set(xlabel="log10(firing frequency)  (-7 = never fired)", ylabel="# latents", title="Feature density")
ax.legend(); plt.tight_layout(); plt.show()

### 4b · Max-activating examples

For a latent, find the tokens where it is most active and print them in context, with the activating token in
⟦brackets⟧. This is the first (and most fallible) step of interpretation: real studies add automated explanations,
ablations, and checks that the explanation *predicts* activations on new text.

In [ ]:
def context(flat_idx, before=10, after=2):
    s, p = divmod(int(flat_idx), SEQ - 1)
    ids = train_seqs[s, 1:].tolist()
    left = tok.decode(ids[max(0, p - before):p]); cur = tok.decode([ids[p]]); right = tok.decode(ids[p + 1:p + 1 + after])
    return (left + "⟦" + cur + "⟧" + right).replace("\n", "↵")

def show_latent(st, j):
    j = int(j)
    print(f"latent {j}: fires on {st['freq'][j] * 100:.2f}% of tokens "
          f"({st['code_rate'][j] * 100:.1f}% of code tokens vs {st['prose_rate'][j] * 100:.1f}% of prose)")
    for v, i in zip(st["top_v"][:, j], st["top_i"][:, j]):
        if v > 0:
            print(f"   {v:6.2f}  {context(i)}")

NAME = f"TopK (k={K})"
st = stats[NAME]
alive = st["count"] >= 10
gap = torch.where(alive, st["code_rate"] - st["prose_rate"], torch.tensor(-1.0))
code_specific = gap.topk(3).indices
prose_specific = torch.where(alive, st["prose_rate"] - st["code_rate"], torch.tensor(-1.0)).topk(2).indices
random_alive = torch.where(alive)[0][torch.randperm(int(alive.sum()), generator=torch.Generator().manual_seed(1))[:3]]

print(f"=== {NAME}: most code-specific latents ===")
for j in code_specific: show_latent(st, j)
print(f"\n=== {NAME}: most prose-specific latents ===")
for j in prose_specific: show_latent(st, j)
print(f"\n=== {NAME}: random live latents (the honest sample) ===")
for j in random_alive: show_latent(st, j)

### 4c · A test that can mislead: neurons vs. latents on "code vs. prose"

A tempting check of the SAE story is: "is there a latent that is more code-specific than any neuron?" Let's run it,
and then think about whether it is the right test.

In [ ]:
neuron_on = acts > 0.1 * acts.std()
n_gap = (neuron_on[tok_is_code].float().mean(0) - neuron_on[~tok_is_code].float().mean(0))
l_gap = gap
print(f"best neuron code-vs-prose firing gap : {n_gap.max():.2f}")
print(f"best SAE latent code-vs-prose gap    : {l_gap.max():.2f}")
print(f"neurons with gap > 0.5: {(n_gap > 0.5).sum().item()} / {D_MLP};  latents with gap > 0.5: {(l_gap > 0.5).sum().item()} / {D_SAE}")

**Why this comparison can come out "wrong" for the SAE.** If, as in our CPU reference run, the best neuron has a
*larger* code-vs-prose gap than any latent, that is not evidence against SAEs. "Is this code?" is a broad, dense
property, and a dense neuron can track it (while also firing for other things: it can be broad *and* polysemantic).
A TopK SAE with k = 32 is pushed toward **specific, sparse** features: "closing parenthesis in Python", "the
`transformer` in `self.transformer`", which fire on a small fraction of code tokens, so their code-vs-prose *rate*
gap is small even though they are perfectly code-specific *when they fire*. Monosemanticity is about each unit
having **one explanation**, not about covering a whole category. Better tests: specificity (when the latent fires,
how often is the explanation true?) and sensitivity (when the explanation holds, how often does it fire?), as in
Bricken et al. Implementing that, with an explanation you write by hand, is a great *Your turn* exercise.

## 5 · (Optional, heavy) a bigger dictionary

With `FULL_SIZE`, the cells above already use 200k tokens, an 8× dictionary (16,384 latents) and 5,000 steps. That
is still tiny by research standards but gives far more interpretable latents. **GPU recommended.** This cell
additionally sweeps $k$ to trace the sparsity/fidelity frontier.

In [ ]:
if heavy_cell("TopK sweep k in {8, 16, 32, 64, 128} to trace the L0-vs-FVE frontier", needs_gpu=False):
    frontier = []
    for k in (8, 16, 32, 64, 128):
        s = SparseAutoencoder(D_MLP, D_SAE, k=k)
        train_sae(s, acts, STEPS // 2, seed=CFG.seed)
        with torch.no_grad():
            xh, _ = s(eval_acts)
            frontier.append((k, 1 - ((xh - eval_acts) ** 2).sum().item() / ((eval_acts - eval_acts.mean(0)) ** 2).sum().item()))
    plt.plot(*zip(*frontier), "o-"); plt.xscale("log"); plt.xlabel("k (= L0)"); plt.ylabel("FVE"); plt.show()

## Honest limitations

* **Scale.** Published SAEs train on 10⁸–10¹⁰ tokens with 10⁴–10⁷ latents. Ours: ~6×10⁴ tokens, 4,096 latents,
  600 steps. Many of our latents will be "partial" or blurry versions of real features.
* **Data.** Shakespeare and Python source are a tiny, odd slice of what the model was trained on (C4 + code), so the
  dictionary only covers that slice.
* **Interpretation by top examples is biased.** Max-activating examples show the *strongest* reason a latent fires,
  not all the reasons. Bricken et al. also check activation specificity and sensitivity across the whole range, and
  use ablations to show the feature affects outputs.
* **Dead latents** in the ReLU SAE could be reduced with resampling (Bricken et al.) or an auxiliary loss (AuxK,
  Gao et al.); we skip both for brevity.

## Your turn

1. **Frontier.** Sweep λ ∈ {5e-4, 1e-3, 2e-3, 4e-3} for ReLU+L1 and k ∈ {8, 16, 32, 64} for TopK, then plot CE-recovered
   vs. L0 for both on one chart. Which architecture dominates at this scale?
2. **Feature steering.** Pick a code-specific latent, add α·(its decoder direction ÷ SCALE) to the MLP output for
   every position of a prose prompt, and look at how the next-token distribution shifts (toward code tokens?).
3. **Dead-latent resampling.** Every 200 steps, re-initialise dead latents' encoder/decoder rows towards inputs the
   SAE reconstructs worst. How does the dead fraction change?
4. **Residual stream.** Train on `blocks.0.hook_resid_post` (512-d) instead of the MLP. Are features similar?

## Beyond the paper — a mini research question

*Do SAE features "split" predictably as the dictionary grows?* Bricken et al. observed **feature splitting**: a
single feature in a small dictionary (say, "Python code") splits into several finer ones in a larger dictionary
("Python imports", "Python function definitions", …). Train TopK SAEs with m ∈ {1024, 4096, 16384} on the same data
and, for each latent of the smallest SAE, find its nearest decoder directions (cosine similarity) in the larger
ones. Do you see clean 1→many splits, and do the children's top activations partition the parent's? Bonus: quantify
how much splitting depends on the amount of training data.

## References

* Elhage, N. et al. (2022). *Toy Models of Superposition.* [arXiv:2209.10652](https://arxiv.org/abs/2209.10652)
* Bricken, T. et al. (2023). *Towards Monosemanticity.* Anthropic. https://transformer-circuits.pub/2023/monosemantic-features/index.html
* Templeton, A. et al. (2024). *Scaling Monosemanticity.* Anthropic. https://transformer-circuits.pub/2024/scaling-monosemanticity/index.html
* Cunningham, H. et al. (2023). *Sparse Autoencoders Find Highly Interpretable Features in Language Models.* [arXiv:2309.08600](https://arxiv.org/abs/2309.08600)
* Gao, L. et al. (2024). *Scaling and evaluating sparse autoencoders.* OpenAI. [arXiv:2406.04093](https://arxiv.org/abs/2406.04093)
* Rajamanoharan, S. et al. (2024). *Improving Dictionary Learning with Gated Sparse Autoencoders.* Google DeepMind. [arXiv:2404.16014](https://arxiv.org/abs/2404.16014)
* Rajamanoharan, S. et al. (2024). *JumpReLU Sparse Autoencoders.* Google DeepMind. [arXiv:2407.14435](https://arxiv.org/abs/2407.14435)
* Lieberum, T. et al. (2024). *Gemma Scope.* Google DeepMind. [arXiv:2408.05147](https://arxiv.org/abs/2408.05147)
* Bloom, J. et al. *SAELens* (MIT). https://github.com/jbloomAus/SAELens